# 03 — Full Pipeline: Baseline → Dataset → GRPO Training

Flusso completo di sviluppo su **Google Colab** con **Unsloth**:

1. **Setup & Imports** — installa dipendenze, verifica GPU
2. **Genera Dataset** — crea il dataset sintetico (5000 samples)
3. **Baseline Evaluation** — valuta il modello base (senza fine-tuning) e logga su wandb
4. **GRPO Training** — addestra il modello con GRPO e logga su wandb
5. **Post-Training Evaluation** — valuta il modello addestrato e confronta con baseline

> **Requisiti:** GPU su Colab (T4 minimo, A100 ideale)

## 1. Setup su Colab

In [1]:
!rm -rf tris
!git clone https://github.com/GiuseppeBellamacina/tris.git
%cd tris
!bash setup.sh

Cloning into 'tris'...
remote: Enumerating objects: 319, done.
remote: Counting objects: 100% (319/319), done.
remote: Compressing objects: 100% (201/201), done.
remote: Total 319 (delta 176), reused 245 (delta 102), pack-reused 0 (from 0)
Receiving objects: 100% (319/319), 603.17 KiB | 16.30 MiB/s, done.
Resolving deltas: 100% (176/176), done.
/content/tris
=== Setup GRPO Strict Generation ===

✅ uv già installato

📦 Installazione dipendenze + progetto...
Using Python 3.12.13 environment at: /usr
Resolved 282 packages in 477ms                                       
Prepared 1 package in 345ms                                              
Uninstalled 1 package in 0.44ms
Installed 1 package in 1msion==0.1.0 (from file:///content/t
 ~ grpo-strict-generation==0.1.0 (from file:///content/tris)
✅ Dipendenze installate + progetto in editable mode

🔍 Verifica installazione...

AMBIENTE GRPO STRICT GENERATION

🔥 PyTorch:
   Version: 2.10.0+cu128
   CUDA available: True
   CUDA version: 12.8
  

## 2. Imports & GPU Check

In [2]:
import os

# vLLM standby mode disabilitata (0): su Colab il runtime forza
# expandable_segments:True nell'allocator CUDA prima dell'avvio del kernel,
# incompatibile con standby mode. Su A100 o runtime custom si può provare "1".
os.environ["UNSLOTH_VLLM_STANDBY"] = "0"

In [3]:
from unsloth import FastLanguageModel

print("Unsloth importato correttamente")

import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:             {torch.cuda.get_device_name(0)}")
    print(f"VRAM:            {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: Nessuna GPU trovata!")

import transformers
import trl

print(f"transformers: {transformers.__version__}")
print(f"trl:          {trl.__version__}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth importato correttamente
PyTorch version: 2.10.0+cu128
CUDA available:  True
GPU:             Tesla T4
VRAM:            15.6 GB
transformers: 4.57.6
trl:          0.22.2


In [4]:
import wandb

# Login a wandb — la prima volta su Colab chiede l'API key.
# Prendi la tua key da https://wandb.ai/authorize
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: cosmico445 (cosmico445-cosmic-inc) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
# Entra nella repo se il clone è appena avvenuto e cwd è ancora /content
import os
from pathlib import Path

_tris = Path.cwd() / "tris"
if _tris.is_dir() and (_tris / "pyproject.toml").exists():
    os.chdir(_tris)
    print(f"cd → {_tris}")
else:
    print(f"cwd già corretto: {Path.cwd()}")

cwd già corretto: /content/tris


In [ ]:
import os
from pathlib import Path

# Resolve repo root robustly:
# - normal: cwd is already /content/tris → pyproject.toml found immediately
# - restarted kernel: cwd is /content → Colab fallback kicks in
def _find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    # Colab-specific fallback: repo always cloned to /content/tris
    colab_path = Path("/content/tris")
    if colab_path.exists():
        return colab_path
    raise RuntimeError("Repo root non trovato. Esegui prima la cella di setup (git clone).")

ROOT = _find_repo_root()
os.chdir(ROOT)

# Costanti
WANDB_PROJECT = "grpo-strict-generation"
DATASET_PATH = ROOT / "data" / "synthetic"
CONFIG_PATH = ROOT / "experiments" / "configs" / "grpo.yaml"

# Esporta il progetto wandb come env var così il Trainer di HuggingFace
# non cade sul default "huggingface" se deve auto-inizializzare wandb.
os.environ["WANDB_PROJECT"] = WANDB_PROJECT

print(f"Root progetto: {ROOT}")

Root progetto: /content/tris


## 3. Genera Dataset Sintetico

Generiamo un dataset serio con **5000 samples** (3 livelli di difficoltà: simple, medium, hard).

In [7]:
from src.datasets.dataloader import load_synthetic_dataset
from src.datasets.synthetic_dataset import generate_dataset

NUM_SAMPLES = 5000
SEED = 42

if DATASET_PATH.exists():
    print(f"Dataset già presente in {DATASET_PATH}, lo ricarico...")
else:
    print(f"Genero {NUM_SAMPLES} samples (seed={SEED})...")
    ds = generate_dataset(num_samples=NUM_SAMPLES, seed=SEED)
    ds.save_to_disk(str(DATASET_PATH))
    print(f"Dataset salvato in {DATASET_PATH}")

ds = load_synthetic_dataset(str(DATASET_PATH))

for split_name, split_ds in ds.items():
    print(f"  {split_name}: {len(split_ds)} samples")
    # Distribuzione per difficoltà
    diffs = split_ds["difficulty"]
    for d in ["simple", "medium", "hard"]:
        count = sum(1 for x in diffs if x == d)
        print(f"    json/{d}: {count}")

Genero 5000 samples (seed=42)...


Saving the dataset (0/1 shards):   0%|          | 0/4000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset salvato in /content/tris/data/synthetic
  train: 4000 samples
    json/simple: 1606
    json/medium: 1417
    json/hard: 977
  test: 1000 samples
    json/simple: 413
    json/medium: 337
    json/hard: 250


## 4. Baseline Evaluation

Valutiamo il modello **base** (senza fine-tuning) sul test set.
Questo ci dà il punto di partenza per misurare il miglioramento dopo GRPO.

In [ ]:
from src.datasets.dataloader import format_prompt_for_model
from src.evaluation.baseline_eval import generate_completions
from src.evaluation.evaluate import compute_detailed_metrics
from src.models.model_loader import load_model_and_tokenizer
from src.utils.config import load_config

config = load_config(str(CONFIG_PATH))

# Carica modello BASE (senza LoRA, senza fast_inference):
# - no "lora" key → no adapters
# - fast_inference=False → no vLLM, no CUDA graph init
# Questo evita il crash del CUDA allocator (use_count > 0) quando
# il modello di training carica vLLM nello stesso processo subito dopo.
base_config = {"model": {**config["model"], "fast_inference": False}}
print(f"Caricamento modello base: {config['model']['name']}")
model, tokenizer = load_model_and_tokenizer(base_config)

Caricamento modello base: Qwen/Qwen2.5-Coder-0.5B-Instruct
==((====))==  Unsloth 2026.3.17: Fast Qwen2 patching. Transformers: 4.57.6. vLLM: 0.18.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [9]:
# Prepara prompt dal test set
test_ds = ds["test"]
MAX_EVAL_SAMPLES = 200  # limita per velocità su Colab
eval_ds = test_ds.select(range(min(MAX_EVAL_SAMPLES, len(test_ds))))

prompts = [format_prompt_for_model(eval_ds[i], tokenizer) for i in range(len(eval_ds))]
task_types = list(eval_ds["task_type"])
difficulties = list(eval_ds["difficulty"])

print(f"Prompts preparati: {len(prompts)}")

# Genera completions
gen_config = {
    "max_new_tokens": 512,
    "temperature": 0.7,
    "top_p": 0.95,
    "do_sample": True,
}

print("Generazione completions baseline...")
baseline_completions = generate_completions(
    model=model,
    tokenizer=tokenizer,
    prompts=prompts,
    generation_config=gen_config,
    num_return_sequences=1,
    batch_size=4,
)
print(f"Completions generate: {len(baseline_completions)}")

Prompts preparati: 200
Generazione completions baseline...


Generating: 100%|██████████| 50/50 [14:26<00:00, 17.33s/it]

Completions generate: 200


In [ ]:
# Calcola metriche baseline
first_completions = [comps[0] for comps in baseline_completions]
baseline_metrics = compute_detailed_metrics(first_completions, task_types, difficulties)

print(f"\n{'='*50}")
print(f"BASELINE — {config['model']['name']}")
print(f"{'='*50}")
print(f"Overall Pass@1: {baseline_metrics['overall_pass_rate']:.4f}")
print("\nPer-category breakdown:")
for cat, stats in baseline_metrics["per_category"].items():
    print(f"  {cat}: {stats['pass_rate']:.4f} ({stats['valid']}/{stats['total']})")

# Log su wandb
wandb.init(
    project=WANDB_PROJECT,
    name=f"baseline-{config['model']['name'].split('/')[-1]}",
    tags=["baseline", "notebook-03"],
    config={"model": config["model"], "eval_samples": len(prompts)},
)

wandb_metrics = {"overall_pass_rate": baseline_metrics["overall_pass_rate"]}
for cat, stats in baseline_metrics["per_category"].items():
    wandb_metrics[f"pass_rate/{cat}"] = stats["pass_rate"]
wandb.log(wandb_metrics)

wandb.finish()
print("\nBaseline loggata su wandb")


BASELINE — Qwen/Qwen2.5-Coder-0.5B-Instruct
Overall Pass@1: 0.9400

Per-category breakdown:
  json/hard: 0.8393 (47/56)
  json/medium: 0.9701 (65/67)
  json/simple: 0.9870 (76/77)


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


overall_pass_rate,▁
pass@1,▁
overall_pass_rate,0.94
pass@1,0.94



Baseline loggata su wandb


In [11]:
# Qualche esempio di completions baseline
from src.training.rewards import combined_reward

print("Esempi di completions baseline:\n")
for i in [0, 1, 5, 10]:
    if i >= len(eval_ds):
        break
    comp = first_completions[i]
    r = combined_reward(comp, task_type="json")
    print(f"--- [{difficulties[i]}] reward={r} ---")
    print(f"Prompt:     {eval_ds[i]['prompt'][:100]}...")
    print(f"Completion: {comp[:200]}")
    print()

Esempi di completions baseline:

--- [medium] reward=1.0 ---
Prompt:     Generate a JSON object representing a model with an "id" (integer), "name" (string), "tags" (array o...
Completion: ```json
{
  "id": 123,
  "name": "Smartphone",
  "tags": ["Android", "iOS"],
  "metadata": {
    "OS": "Android",
    "Price": 999.99,
    "Screen Size": 6.5,
    "Battery Life": 12000
  }
}
```

--- [hard] reward=0.0 ---
Prompt:     Generate a JSON object representing a database migration plan with 3 operations. Each operation shou...
Completion: ```json
{
  "operations": [
    {
      "order": 1,
      "type": "create_table",
      "table": "users",
      "definition": {
        "columns": [
          {"name": "id", "type": "int", "order": 1}

--- [medium] reward=1.0 ---
Prompt:     Generate a JSON object representing a dataset with an "id" (integer), "name" (string), "tags" (array...
Completion: ```json
{
  "id": 1,
  "name": "Sample Dataset",
  "tags": ["Data Analysis", "Machine Learning"],
  "me

In [12]:
# Libera VRAM dal modello base
del model
torch.cuda.empty_cache()
print("VRAM liberata")

VRAM liberata


## 5. GRPO Training

Addestriamo il modello con **GRPO** (Group Relative Policy Optimization).
Il config viene letto da `experiments/configs/grpo.yaml`.

Parametri chiave:
- **learning_rate**: 5e-6
- **optim**: paged_adamw_8bit (meno VRAM)
- **max_grad_norm**: 0.1 (gradient clipping aggressivo)
- **num_generations**: 8 (gruppo per calcolare i vantaggi relativi)
- **beta**: 0.04 (KL penalty)

> **fast_inference**: se `true` nel config, usa vLLM interno per generazione 2-5x più veloce.
> Si attiva automaticamente solo su Linux con vLLM installato; altrimenti fallback a `model.generate()`.

> **UNSLOTH_VLLM_STANDBY**: disabilitato su Colab T4 perché il runtime forza
> `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` prima dell'avvio del kernel,
> incompatibile con la standby mode. Su A100 o runtime custom potrebbe funzionare.


In [13]:
from datasets import Dataset
from src.training.rewards import build_reward_function

# Ricarica modello CON LoRA per il training
print(f"Caricamento modello con LoRA: {config['model']['name']}")
model, tokenizer = load_model_and_tokenizer(config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Parametri totali:    {total:,}")
print(f"Parametri trainable: {trainable:,} ({100 * trainable / total:.2f}%)")

Caricamento modello con LoRA: Qwen/Qwen2.5-Coder-0.5B-Instruct
fast_inference abilitato (vLLM backend)
INFO 03-28 11:38:35 [vllm_utils.py:724] Unsloth: Patching vLLM v1 graph capture
==((====))==  Unsloth 2026.3.17: Fast Qwen2 patching. Transformers: 4.57.6. vLLM: 0.18.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/qwen2.5-coder-0.5b-instruct-bnb-4bit with actual GPU utilization = 71.61%
Unsloth: Your GPU has CUDA compute capability 7.5 with VRAM = 14.56 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 2048. Num Sequences = 48.
Unsloth: vLLM's KV Cache can use up to 9.8 GB. Also swap space = 0 GB.
Unsloth: Not a

<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.


INFO 03-28 11:38:46 [weight_utils.py:618] No model.safetensors.index.json found in remote.


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


...

INFO 03-28 11:44:40 [gpu_worker.py:617] CUDA graph pool memory: 0.56 GiB (actual), 0.76 GiB (estimated), difference: 0.2 GiB (35.3%).
INFO 03-28 11:44:40 [core.py:281] init engine (profile, create kv cache, warmup model) took 352.56 seconds
INFO 03-28 11:44:42 [llm.py:391] Supported tasks: ('generate',)
Unsloth: Just some info: will skip parsing ['post_feedforward_layernorm', 'post_layernorm', 'k_norm', 'norm1', 'pre_feedforward_layernorm', 'input_layernorm', 'q_norm', 'layer_norm1', 'layer_norm2', 'attention_norm', 'norm2', 'ffn_norm', 'post_attention_layernorm', 'norm']


Some weights of Qwen2ForCausalLM were not initialized from the model checkpoint at unsloth/qwen2.5-coder-0.5b-instruct-bnb-4bit and are newly initialized: ['lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Performing substitution for additional_keys=set()
Unsloth: Just some info: will skip parsing ['post_feedforward_layernorm', 'post_layernorm', 'k_norm', 'norm1', 'pre_feedforward_layernorm', 'input_layernorm', 'q_norm', 'layer_norm1', 'layer_norm2', 'attention_norm', 'norm2', 'cross_attn_post_attention_layernorm', 'ffn_norm', 'cross_attn_input_layernorm', 'post_attention_layernorm', 'norm']


Unsloth 2026.3.17 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


Parametri totali:    323,917,696
Parametri trainable: 8,798,208 (2.72%)


In [14]:
# Prepara dataset di training (prompt formattati)
train_ds = ds["train"]
formatted = []
for i in range(len(train_ds)):
    s = train_ds[i]
    p = format_prompt_for_model(s, tokenizer)
    formatted.append({"prompt": p})

prompt_dataset = Dataset.from_list(formatted)
print(f"Training dataset: {len(prompt_dataset)} prompt pronti")

# Reward function
reward_fn = build_reward_function(config["reward"])
print("Reward function creata")

Training dataset: 4000 prompt pronti
Reward function creata


In [ ]:
import datetime

from trl import GRPOConfig

training_cfg = config["training"]
grpo_cfg = config["grpo"]

output_dir = str(ROOT / "experiments" / "checkpoints" / "grpo_full")
log_dir = str(ROOT / "experiments" / "logs" / "grpo_full")
Path(output_dir).mkdir(parents=True, exist_ok=True)
Path(log_dir).mkdir(parents=True, exist_ok=True)

_ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = f"grpo-full-{_ts}"

# bf16 vs fp16 in base alla GPU
_supports_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
use_bf16 = _supports_bf16
use_fp16 = not _supports_bf16 and torch.cuda.is_available()

grpo_config = GRPOConfig(
    output_dir=output_dir,
    run_name=run_name,
    # Training
    max_steps=training_cfg.get("max_steps", 1000),
    per_device_train_batch_size=training_cfg.get("per_device_train_batch_size", 1),
    gradient_accumulation_steps=training_cfg.get("gradient_accumulation_steps", 8),
    learning_rate=training_cfg.get("learning_rate", 5e-6),
    lr_scheduler_type=training_cfg.get("lr_scheduler_type", "cosine"),
    warmup_ratio=training_cfg.get("warmup_ratio", 0.1),
    optim=training_cfg.get("optim", "paged_adamw_8bit"),
    weight_decay=training_cfg.get("weight_decay", 0.1),
    max_grad_norm=training_cfg.get("max_grad_norm", 0.1),
    bf16=use_bf16,
    fp16=use_fp16,
    # GRPO
    num_generations=grpo_cfg.get("num_generations", 8),
    max_completion_length=grpo_cfg.get("max_completion_length", 512),
    max_prompt_length=grpo_cfg.get("max_prompt_length", 256),
    beta=grpo_cfg.get("beta", 0.04),
    temperature=grpo_cfg.get("temperature", 0.7),
    # Logging
    logging_dir=log_dir,
    logging_steps=training_cfg.get("logging_steps", 10),
    save_steps=training_cfg.get("save_steps", 200),
    save_total_limit=training_cfg.get("save_total_limit", 3),
    report_to="wandb",
)

# Inizializza il run wandb PRIMA del trainer, così il GRPOTrainer
# riusa il run attivo anziché crearne uno con project="huggingface".
wandb.init(
    project=WANDB_PROJECT,
    name=run_name,
    tags=["grpo", "notebook-03", config["model"]["name"].split("/")[-1]],
    config=config,
)

print("GRPOConfig creata")
print(f"  run_name:       {grpo_config.run_name}")
print(f"  max_steps:      {grpo_config.max_steps}")
print(f"  num_generations: {grpo_config.num_generations}")
print(f"  batch_size:     {grpo_config.per_device_train_batch_size}")
print(f"  grad_accum:     {grpo_config.gradient_accumulation_steps}")
print(f"  learning_rate:  {grpo_config.learning_rate}")
print(f"  optim:          {grpo_config.optim}")
print(f"  beta:           {grpo_config.beta}")
print(f"  bf16={use_bf16}, fp16={use_fp16}")

GRPOConfig creata
  run_name:       grpo-full-20260328_114721
  max_steps:      1000
  num_generations: 8
  batch_size:     1
  grad_accum:     8
  learning_rate:  5e-06
  optim:          OptimizerNames.PAGED_ADAMW_8BIT
  beta:           0.04
  bf16=False, fp16=True


In [ ]:
from trl import GRPOTrainer

trainer = GRPOTrainer(
    model=model,
    args=grpo_config,
    train_dataset=prompt_dataset,
    reward_funcs=reward_fn,
    processing_class=tokenizer,
)

print(f"Trainer inizializzato (tipo: {type(trainer).__name__})")
print(f"\nAvvio GRPO training ({grpo_config.max_steps} step)...\n")

trainer.train()
print("\nTraining completato!")

Trainer inizializzato (tipo: UnslothGRPOTrainer)

Avvio GRPO training (1000 step)...



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,000 | Num Epochs = 1 | Total steps = 1,000
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / reward_fn / mean,rewards / reward_fn / std
10,0.000000,0.862500,0.081646,170.475000,145.100000,208.300000,0.137500,115.260715,93.900000,155.600000,0.000005,0.862500,0.081646
20,0.000000,0.912500,0.174228,196.075000,115.200000,261.800000,0.075000,183.050003,115.200000,248.400000,0.000005,0.912500,0.174228
30,0.000000,0.975000,0.046291,114.962500,83.600000,161.700000,0.025000,108.929169,83.600000,146.200000,0.000007,0.975000,0.046291
40,0.000000,0.981250,0.053033,114.400000,84.300000,183.100000,0.012500,108.623214,84.300000,140.600000,0.000006,0.981250,0.053033
50,0.000000,0.950000,0.117002,157.750000,107.700000,233.600000,0.050000,148.500003,107.700000,197.900000,0.000007,0.950000,0.117002
60,0.000000,0.978125,0.061872,116.325000,80.300000,174.100000,0.012500,112.487502,80.300000,163.500000,0.000006,0.978125,0.061872
70,0.000000,0.959375,0.073645,150.812500,101.000000,244.400000,0.025000,143.733930,101.000000,215.600000,0.000013,0.959375,0.073645
80,0.000000,0.975000,0.046291,117.862500,84.500000,179.200000,0.025000,112.591669,84.500000,162.500000,0.000011,0.975000,0.046291
90,0.000000,0.975000,0.070711,118.612500,75.400000,209.300000,0.025000,110.508931,75.400000,181.700000,0.000014,0.975000,0.070711
100,0.000000,0.950000,0.087110,124.600000,79.600000,176.700000,0.037500,114.490001,79.600000,157.700000,0.000021,0.950000,0.087110


Unsloth: Will smartly offload gradients to save VRAM!


In [ ]:
# Salva il modello finale
final_path = Path(output_dir) / "final"
trainer.save_model(str(final_path))
tokenizer.save_pretrained(str(final_path))
print(f"Modello salvato in {final_path}")

# Chiudi il run wandb del training prima di aprirne uno nuovo per l'eval
wandb.finish()
print("Run wandb training chiuso")

## 6. Post-Training Evaluation

Valutiamo il modello addestrato sullo **stesso test set** usato per la baseline.

In [ ]:
# Genera completions con il modello trainato
# Usiamo gli stessi prompt della baseline per un confronto diretto
print("Generazione completions post-GRPO...")
grpo_completions = generate_completions(
    model=model,
    tokenizer=tokenizer,
    prompts=prompts,
    generation_config=gen_config,
    num_return_sequences=1,
    batch_size=4,
)

# Calcola metriche post-GRPO
grpo_first = [comps[0] for comps in grpo_completions]
grpo_metrics = compute_detailed_metrics(grpo_first, task_types, difficulties)

print(f"\n{'='*50}")
print(f"POST-GRPO — {config['model']['name']}")
print(f"{'='*50}")
print(f"Overall Pass@1: {grpo_metrics['overall_pass_rate']:.4f}")
print("\nPer-category breakdown:")
for cat, stats in grpo_metrics["per_category"].items():
    print(f"  {cat}: {stats['pass_rate']:.4f} ({stats['valid']}/{stats['total']})")

# Log su wandb
wandb.init(
    project=WANDB_PROJECT,
    name=f"grpo-eval-{config['model']['name'].split('/')[-1]}",
    tags=["grpo-eval", "notebook-03"],
    config=config,
)

grpo_wandb = {"overall_pass_rate": grpo_metrics["overall_pass_rate"]}
for cat, stats in grpo_metrics["per_category"].items():
    grpo_wandb[f"pass_rate/{cat}"] = stats["pass_rate"]
wandb.log(grpo_wandb)

wandb.finish()
print("\nMetriche post-GRPO loggate su wandb")

## 7. Confronto Baseline vs GRPO

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

all_cats = sorted(set(
    list(baseline_metrics["per_category"].keys()) +
    list(grpo_metrics["per_category"].keys())
))
labels = ["overall"] + all_cats
b_values = [baseline_metrics["overall_pass_rate"]] + [
    baseline_metrics["per_category"].get(c, {}).get("pass_rate", 0.0) for c in all_cats
]
g_values = [grpo_metrics["overall_pass_rate"]] + [
    grpo_metrics["per_category"].get(c, {}).get("pass_rate", 0.0) for c in all_cats
]
deltas = [g - b for g, b in zip(g_values, b_values)]

x = np.arange(len(labels))
width = 0.32

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f"Baseline vs Post-GRPO — {config['model']['name']}", fontsize=13, fontweight="bold")

# ── Grouped bar: pass rate per categoria ──────────────────────────────────────
ax = axes[0]
bars_b = ax.bar(x - width / 2, b_values, width, label="Baseline", color="#4C72B0", alpha=0.85)
bars_g = ax.bar(x + width / 2, g_values, width, label="Post-GRPO", color="#DD8452", alpha=0.85)
ax.set_ylim(0, 1.15)
ax.set_ylabel("Pass@1")
ax.set_title("Pass Rate per Category")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=20, ha="right")
ax.legend()
ax.bar_label(bars_b, fmt="%.2f", padding=3, fontsize=8)
ax.bar_label(bars_g, fmt="%.2f", padding=3, fontsize=8)
ax.axhline(0, color="black", linewidth=0.5)

# ── Delta bar: miglioramento per categoria ────────────────────────────────────
ax2 = axes[1]
colors = ["#2ca02c" if d >= 0 else "#d62728" for d in deltas]
bars_d = ax2.bar(x, deltas, width * 1.8, color=colors, alpha=0.85)
ax2.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax2.set_ylabel("Δ Pass@1 (GRPO − Baseline)")
ax2.set_title("Delta per Category")
ax2.set_xticks(x)
ax2.set_xticklabels(labels, rotation=20, ha="right")
ax2.bar_label(bars_d, fmt="%+.3f", padding=3, fontsize=8)

fig.tight_layout()
plt.show()

# Tabella testuale di riepilogo
print(f"\n{'Categoria':<30} {'Baseline':>10} {'Post-GRPO':>10} {'Delta':>10}")
print("-" * 62)
for lbl, b, g, d in zip(labels, b_values, g_values, deltas):
    print(f"  {lbl:<28} {b:>10.4f} {g:>10.4f} {d:>+10.4f}")
delta_overall = g_values[0] - b_values[0]
print()
if delta_overall > 0:
    print(f"GRPO ha migliorato il pass rate di {delta_overall:+.4f} ({delta_overall / max(b_values[0], 1e-6) * 100:+.1f}%)")
elif delta_overall < 0:
    print(f"GRPO ha peggiorato il pass rate di {delta_overall:.4f}")
else:
    print("Nessun cambiamento")


In [ ]:
# Qualche esempio di completions post-GRPO
print("Esempi di completions post-GRPO:\n")
for i in [0, 1, 5, 10]:
    if i >= len(eval_ds):
        break
    comp = grpo_first[i]
    r = combined_reward(comp, task_type="json")
    print(f"--- [{difficulties[i]}] reward={r} ---")
    print(f"Prompt:     {eval_ds[i]['prompt'][:100]}...")
    print(f"Completion: {comp[:200]}")
    print()

In [ ]:
# Pulizia VRAM
del trainer
del model
torch.cuda.empty_cache()
print("VRAM liberata")

## Riepilogo

Questo notebook ha eseguito il flusso completo:

1. **Dataset** — 5000 samples JSON (simple/medium/hard)
2. **Baseline** — valutazione modello base → metriche su wandb
3. **GRPO Training** — addestramento con reward rule-based → loss/reward su wandb
4. **Post-GRPO** — valutazione modello trainato → confronto con baseline

Tutti i run sono su **wandb** sotto il progetto `grpo-strict-generation`.

**Comandi CLI equivalenti:**
```bash
# Baseline
uv run python -m src.evaluation.baseline_eval --config experiments/configs/baseline.yaml

# GRPO Training
uv run python -m src.training.grpo_train --config experiments/configs/grpo.yaml
```